In [ ]:
%matplotlib widget

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from ipywidgets import FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

mpl.rcParams['figure.max_open_warning'] = 50

# ------------------------------------------------------------
# CONTROLS
# ------------------------------------------------------------

filter_title = HTML(value="<b>Filter Type:</b>")
filter_radio = RadioButtons(options=['Low-pass', 'High-pass'], value='Low-pass', description='', layout=Layout(width='150px'))

rc_title = HTML(value="<b>RC Filter</b>")
rl_title = HTML(value="<b>RL Filter</b>")

R_RC = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='R_RC (kΩ):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))
C_RC = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='C (μF):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))

R_RL = FloatSlider(min=0.1, max=10.0, step=0.1, value=2.0, description='R_RL (kΩ):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))
L_RL = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='L (H):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))

# ------------------------------------------------------------
# CONTROL LAYOUT
# ------------------------------------------------------------

filter_box = VBox([filter_title, filter_radio], layout=Layout(width='180px'))

rc_box = VBox([rc_title, R_RC, C_RC], layout=Layout(width='340px'))
rl_box = VBox([rl_title, R_RL, L_RL], layout=Layout(width='340px'))

control_row = HBox([rc_box, rl_box], layout=Layout(width='720px', justify_content='space-between', align_items='flex-start'))

# ------------------------------------------------------------
# NUMERICAL OUTPUT
# ------------------------------------------------------------

parameter_title = HTML(value="<b>Calculated Filter Parameters</b>")

tau_output = HTMLMath(layout=Layout(width='800px'))
omega_output = HTMLMath(layout=Layout(width='800px'))
frequency_output = HTMLMath(layout=Layout(width='800px'))
error_output = HTMLMath(layout=Layout(width='800px'))
status_output = HTML(layout=Layout(width='800px'))

parameter_box = VBox([parameter_title, tau_output, omega_output, frequency_output, error_output, status_output], layout=Layout(width='820px'))

# ------------------------------------------------------------
# CREATE TWO INDEPENDENT FIGURES
# ------------------------------------------------------------

plt.ioff()

fig_mag, ax_mag = plt.subplots(figsize=(5.8, 4.0))
fig_phase, ax_phase = plt.subplots(figsize=(5.8, 4.0))

# ------------------------------------------------------------
# INITIAL PARAMETERS
# ------------------------------------------------------------

Rrc_initial = R_RC.value * 1e3
C_initial = C_RC.value * 1e-6

Rrl_initial = R_RL.value * 1e3
L_initial = L_RL.value

tau_RC_initial = Rrc_initial * C_initial
tau_RL_initial = L_initial / Rrl_initial

wc_RC_initial = 1.0 / tau_RC_initial
wc_RL_initial = 1.0 / tau_RL_initial

# ------------------------------------------------------------
# FREQUENCY RANGE
# ------------------------------------------------------------

wc_min_initial = min(wc_RC_initial, wc_RL_initial)
wc_max_initial = max(wc_RC_initial, wc_RL_initial)

omega_initial = np.logspace(np.log10(wc_min_initial / 100.0), np.log10(wc_max_initial * 100.0), 1200)

# ------------------------------------------------------------
# RESPONSE FUNCTION
# ------------------------------------------------------------

def calculate_response(filter_type, omega, tau_RC, tau_RL):

    jw = 1j * omega

    if filter_type == 'Low-pass':
        H_RC = 1.0 / (1.0 + jw * tau_RC)
        H_RL = 1.0 / (1.0 + jw * tau_RL)

    else:
        H_RC = jw * tau_RC / (1.0 + jw * tau_RC)
        H_RL = jw * tau_RL / (1.0 + jw * tau_RL)

    magnitude_RC = 20.0 * np.log10(np.maximum(np.abs(H_RC), 1e-12))
    magnitude_RL = 20.0 * np.log10(np.maximum(np.abs(H_RL), 1e-12))

    phase_RC = np.degrees(np.angle(H_RC))
    phase_RL = np.degrees(np.angle(H_RL))

    return magnitude_RC, magnitude_RL, phase_RC, phase_RL

# ------------------------------------------------------------
# INITIAL RESPONSES
# ------------------------------------------------------------

mag_RC_initial, mag_RL_initial, phase_RC_initial, phase_RL_initial = calculate_response(filter_radio.value, omega_initial, tau_RC_initial, tau_RL_initial)

# ------------------------------------------------------------
# MAGNITUDE PLOT
# ------------------------------------------------------------

line_mag_RC, = ax_mag.semilogx(omega_initial, mag_RC_initial, linewidth=2.0, label='RC filter')
line_mag_RL, = ax_mag.semilogx(omega_initial, mag_RL_initial, '--', linewidth=2.0, label='RL filter')

wc_line_mag_RC = ax_mag.axvline(wc_RC_initial, linestyle=':', linewidth=1.2)
wc_line_mag_RL = ax_mag.axvline(wc_RL_initial, linestyle=':', linewidth=1.2)

ax_mag.axhline(-3.0103, linestyle=':', linewidth=1.0)

ax_mag.set_title('Magnitude Response')
ax_mag.set_xlabel('Angular Frequency ω (rad/s)')
ax_mag.set_ylabel('Magnitude (dB)')
ax_mag.set_ylim(-60, 5)
ax_mag.grid(True, which='both', linestyle=':', alpha=0.7)
ax_mag.legend(loc='best')

# ------------------------------------------------------------
# PHASE PLOT
# ------------------------------------------------------------

line_phase_RC, = ax_phase.semilogx(omega_initial, phase_RC_initial, linewidth=2.0, label='RC filter')
line_phase_RL, = ax_phase.semilogx(omega_initial, phase_RL_initial, '--', linewidth=2.0, label='RL filter')

wc_line_phase_RC = ax_phase.axvline(wc_RC_initial, linestyle=':', linewidth=1.2)
wc_line_phase_RL = ax_phase.axvline(wc_RL_initial, linestyle=':', linewidth=1.2)

phase_reference = ax_phase.axhline(-45.0, linestyle=':', linewidth=1.0)

ax_phase.set_title('Phase Response')
ax_phase.set_xlabel('Angular Frequency ω (rad/s)')
ax_phase.set_ylabel('Phase (deg)')
ax_phase.set_ylim(-100, 10)
ax_phase.grid(True, which='both', linestyle=':', alpha=0.7)
ax_phase.legend(loc='best')

# ------------------------------------------------------------
# AXIS FORMAT
# ------------------------------------------------------------

for ax in [ax_mag, ax_phase]:

    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_frame_on(True)

    for spine in ['left', 'right', 'top', 'bottom']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(0.8)
        ax.spines[spine].set_clip_on(False)

for fig in [fig_mag, fig_phase]:
    fig.subplots_adjust(left=0.14, right=0.96, bottom=0.16, top=0.88)

# ------------------------------------------------------------
# CANVAS SETTINGS
# ------------------------------------------------------------

for canvas in [fig_mag.canvas, fig_phase.canvas]:
    canvas.toolbar_visible = False
    canvas.header_visible = False
    canvas.footer_visible = False
    canvas.resizable = False
    canvas.layout.width = '580px'
    canvas.layout.height = '400px'
    canvas.layout.overflow = 'visible'

# ------------------------------------------------------------
# PLOT LAYOUT
# ------------------------------------------------------------

mag_box = VBox([fig_mag.canvas], layout=Layout(width='580px', overflow='visible'))
phase_box = VBox([fig_phase.canvas], layout=Layout(width='580px', overflow='visible'))

plot_row = HBox([mag_box, phase_box], layout=Layout(width='1180px', justify_content='space-between', align_items='center', overflow='visible'))

# ------------------------------------------------------------
# UPDATE PARAMETER OUTPUT
# ------------------------------------------------------------

def update_parameters():

    Rrc = R_RC.value * 1e3
    C = C_RC.value * 1e-6

    Rrl = R_RL.value * 1e3
    L = L_RL.value

    tau_RC = Rrc * C
    tau_RL = L / Rrl

    wc_RC = 1.0 / tau_RC
    wc_RL = 1.0 / tau_RL

    fc_RC = wc_RC / (2.0 * np.pi)
    fc_RL = wc_RL / (2.0 * np.pi)

    equivalence_error = 100.0 * abs(tau_RC - tau_RL) / max(tau_RC, tau_RL)

    tau_output.value = rf"$$\tau_{{RC}}=R_{{RC}}C={tau_RC:.6f}\ \mathrm{{s}}\qquad\qquad\tau_{{RL}}=\frac{{L}}{{R_{{RL}}}}={tau_RL:.6f}\ \mathrm{{s}}$$"

    omega_output.value = rf"$$\omega_{{c,RC}}=\frac{{1}}{{\tau_{{RC}}}}={wc_RC:.2f}\ \mathrm{{rad/s}}\qquad\qquad\omega_{{c,RL}}=\frac{{1}}{{\tau_{{RL}}}}={wc_RL:.2f}\ \mathrm{{rad/s}}$$"

    frequency_output.value = rf"$$f_{{c,RC}}={fc_RC:.2f}\ \mathrm{{Hz}}\qquad\qquad f_{{c,RL}}={fc_RL:.2f}\ \mathrm{{Hz}}$$"

    error_output.value = rf"$$\mathrm{{Equivalence\ Error}}=100\frac{{|\tau_{{RC}}-\tau_{{RL}}|}}{{\max(\tau_{{RC}},\tau_{{RL}})}}={equivalence_error:.2f}\%$$"

    if np.isclose(tau_RC, tau_RL, rtol=0.0, atol=1e-12):
        status_output.value = "<div style='font-size:15px;'><b>RC and RL filters are equivalent: τ<sub>RC</sub> = τ<sub>RL</sub></b></div>"
    else:
        status_output.value = "<div style='font-size:15px;'><b>Adjust the circuit parameters until the Equivalence Error becomes 0%.</b></div>"

# ------------------------------------------------------------
# UPDATE PLOTS
# ------------------------------------------------------------

def update_plots():

    Rrc = R_RC.value * 1e3
    C = C_RC.value * 1e-6

    Rrl = R_RL.value * 1e3
    L = L_RL.value

    tau_RC = Rrc * C
    tau_RL = L / Rrl

    wc_RC = 1.0 / tau_RC
    wc_RL = 1.0 / tau_RL

    wc_min = min(wc_RC, wc_RL)
    wc_max = max(wc_RC, wc_RL)

    omega = np.logspace(np.log10(wc_min / 100.0), np.log10(wc_max * 100.0), 1200)

    mag_RC, mag_RL, phase_RC, phase_RL = calculate_response(filter_radio.value, omega, tau_RC, tau_RL)

    # --------------------------------------------------------
    # UPDATE DATA ONLY
    # --------------------------------------------------------

    line_mag_RC.set_data(omega, mag_RC)
    line_mag_RL.set_data(omega, mag_RL)

    line_phase_RC.set_data(omega, phase_RC)
    line_phase_RL.set_data(omega, phase_RL)

    wc_line_mag_RC.set_xdata([wc_RC, wc_RC])
    wc_line_mag_RL.set_xdata([wc_RL, wc_RL])

    wc_line_phase_RC.set_xdata([wc_RC, wc_RC])
    wc_line_phase_RL.set_xdata([wc_RL, wc_RL])

    # --------------------------------------------------------
    # UPDATE AXES
    # --------------------------------------------------------

    xmin = wc_min / 100.0
    xmax = wc_max * 100.0

    ax_mag.set_xlim(xmin, xmax)
    ax_phase.set_xlim(xmin, xmax)

    if filter_radio.value == 'Low-pass':
        phase_reference.set_ydata([-45.0, -45.0])
        ax_phase.set_ylim(-100, 10)

    else:
        phase_reference.set_ydata([45.0, 45.0])
        ax_phase.set_ylim(-10, 100)

    ax_mag.set_ylim(-60, 5)

    # --------------------------------------------------------
    # REDRAW EXISTING CANVASES ONLY
    # --------------------------------------------------------

    fig_mag.canvas.draw_idle()
    fig_phase.canvas.draw_idle()

# ------------------------------------------------------------
# CALLBACKS
# ------------------------------------------------------------

def slider_changed(change):

    update_parameters()
    update_plots()

def filter_changed(change):

    update_parameters()
    update_plots()

R_RC.observe(slider_changed, names='value')
C_RC.observe(slider_changed, names='value')
R_RL.observe(slider_changed, names='value')
L_RL.observe(slider_changed, names='value')

filter_radio.observe(filter_changed, names='value')

# ------------------------------------------------------------
# INITIALIZATION
# ------------------------------------------------------------

update_parameters()
update_plots()

# ------------------------------------------------------------
# NOTEBOOK OUTPUT STYLE
# ------------------------------------------------------------

display(HTML("""
<style>
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
}
.jp-OutputArea-child {
    overflow: visible !important;
    max-height: none !important;
}
.jp-OutputArea {
    overflow: visible !important;
    max-height: none !important;
}
</style>
"""))

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(HTML("<h3>First-Order RC–RL Filter Comparison</h3>"))
display(filter_box)
display(control_row)
display(parameter_box)
display(plot_row)

for fig in [fig_mag, fig_phase]:
    fig.canvas.draw()